**Load Data**

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from scipy.stats import gmean
from sklearn.impute import KNNImputer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import StackingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import BaggingRegressor
import itertools
import plotly.graph_objects as go

In [2]:
wqi_data = pd.read_csv('quang_ninh_wqi_v4.csv')

# 1. Data Pre-processing


---






In [3]:
print(wqi_data.columns)

Index(['Location', 'ID', 'Month', 'Temp', 'pH', 'DO', 'Sal', 'Turb', 'TSS',
       'COD', 'BOD5', 'SO4', 'NO3', 'NO2', 'NH4', 'PO4', 'MBAS', 'Cyan',
       'Fluor', 'Cl', 'OilFat', 'Coliform', 'EColi', 'As', 'Hg', 'Pb', 'Cd',
       'Fe', 'Mn', 'Cu', 'Ni', 'Zn', 'Cr', 'CrVI', 'Phenol', 'RadAlpha',
       'RadBeta', 'Aldrin', 'BHC', 'Dieldrin', 'DDTs', 'Heptach', 'TOC'],
      dtype='object')


Location: Sampling location

Sample_ID: Unique sample identifier

Month: Monitoring month

Temperature_C: Water temperature in Celsius

pH: pH level

DO_mg_L: Dissolved Oxygen in mg/L

Salinity: Salinity of water

Turbidity: Water turbidity

TSS_mg_L: Total Suspended Solids in mg/L

COD_mg_L: Chemical Oxygen Demand in mg/L

BOD5_mg_L: Biochemical Oxygen Demand over 5 days in mg/L

SO4_mg_L: Sulfate concentration in mg/L

NO3_mg_L: Nitrate concentration in mg/L

NO2_mg_L: Nitrite concentration in mg/L

NH4_mg_L: Ammonium concentration in mg/L

PO4_mg_L: Phosphate concentration in mg/L

MBAS: Surfactant content (MBAS method)

Cyanide: Cyanide concentration

Fluoride: Fluoride concentration

Cl_mg_L: Chloride concentration in mg/L

Total_Oil_Fat_mg_L: Total oil and fat content in mg/L

Coliform_MPN_100ml: Coliform bacteria count (MPN/100ml)

EColi: E. Coli bacteria count

As_mg_L: Arsenic concentration in mg/L

Hg_mg_L: Mercury concentration in mg/L

Pb_mg_L: Lead concentration in mg/L

Cd_mg_L: Cadmium concentration in mg/L

Fe_mg_L: Iron concentration in mg/L

Mn: Manganese concentration

Cu: Copper concentration

Ni: Nickel concentration

Zn: Zinc concentration

Cr: Chromium concentration

CrVI: Hexavalent Chromium concentration

Total_Phenol: Total phenol content

Total_Rad_Alpha: Total alpha radioactivity

Total_Rad_Beta: Total beta radioactivity

Aldrin: Aldrin pesticide concentration

BHC: BHC pesticide concentration

Dieldrin: Dieldrin pesticide concentration

DDTs: DDT and its derivatives concentration

Heptachlor_Proxide: Heptachlor & Heptachloreproxide concentration

TOC: Total Organic Carbon


In [4]:
wqi_data_droped = wqi_data.drop([
    'Location', 'Sample_ID', 'Month', 'Salinity', 'Turbidity', 'TSS_mg_L', 'SO4_mg_L', 'MBAS', 'Cyanide', 'Fluoride',
    'Cl_mg_L', 'Total_Oil_Fat_mg_L', 'Fe_mg_L', 'Mn', 'Cu', 'Ni', 'Zn', 'Total_Phenol',
    'Total_Rad_Alpha', 'Total_Rad_Beta'
], axis=1)

KeyError: "['Sample_ID', 'Salinity', 'Turbidity', 'TSS_mg_L', 'SO4_mg_L', 'Cyanide', 'Fluoride', 'Cl_mg_L', 'Total_Oil_Fat_mg_L', 'Fe_mg_L', 'Total_Phenol', 'Total_Rad_Alpha', 'Total_Rad_Beta'] not found in axis"

Drop some columns which don't impact to WQI scores

## 1.1 Missing Values


---
Identify and handle missing values.

In [ ]:
wqi_data_filled = wqi_data_droped.replace('X', np.nan)
wqi_data_filled = wqi_data_droped.replace('x', np.nan)
wqi_data_filled.isnull().sum()

Some cells contain 'x' instead of being empty, so the missing values were standardized and the number of missing values was calculated.


In [ ]:
wqi_data_droped_v2 = wqi_data_filled.drop([
    'Cr', 'Aldrin', 'BHC', 'Dieldrin', 'DDTs', 'Heptachlor_Proxide', 'TOC'
], axis=1)

print(wqi_data_droped_v2.head())

Columns with about 75% missing values were removed.

In [ ]:
wqi_data_cleaned = wqi_data_droped_v2[~wqi_data_droped_v2.apply(lambda row: row.astype(str).str.contains(r'\s*KCM\s*', regex=True).any(), axis=1)]

KCM: Value could not be measured; these records were removed from the dataset

## 1.2 Column, Cell data type conversion

In [ ]:
def handle_less_than_numeric(val):
    # If the value is a string
    if isinstance(val, str):
        val = val.strip()
        # If it matches the pattern "<x" or "<x.y"
        match = re.fullmatch(r'<\s*([0-9,]*\.?[0-9]+)', val)
        if match:
            x_str = match.group(1).replace(',', '.')
            return float(x_str)
        # If it's a numeric string with comma or dot
        try:
            return float(val.replace(',', '.'))
        except ValueError:
            return val
    # If it's already a number, keep as is
    elif isinstance(val, (int, float)):
        return val
    # Other values remain unchanged
    return val

def convert_column_type(col):
    # Remove NaN to check
    non_na = col.dropna()

    # Check if all are numbers
    if non_na.apply(lambda x: isinstance(x, (int, float))).all():
        return col.astype(float)
    else:
        return col.astype('category')


# Apply the conversion to handle "<" and numeric strings
wqi_data_cleaned_v2 = wqi_data_cleaned.applymap(handle_less_than_numeric)

# Convert all to numeric (if not a number → NaN)
wqi_data_convert_cell = wqi_data_cleaned_v2.apply(convert_column_type)

print(wqi_data_convert_cell.head())

Cells with values less than x will be converted to type x

In [ ]:
wqi_data_convert_cell.isnull().sum()

There are 2 columns which include missing values that are EColi and CrVI

In [ ]:
# Select numeric columns
numeric_cols = wqi_data_convert_cell.select_dtypes(include=['int64', 'float64']).columns
df_numeric = wqi_data_convert_cell[numeric_cols]

# Compute the correlation matrix
corr_matrix = df_numeric.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Heatmap: Relationships among numeric variables in the WQI dataset")
plt.show()

Because EColi and CrVI do not have any meaningful relationship with the other features, we cannot apply linear regression to predict them.

In [ ]:
# List of numeric columns to impute
num_cols_to_impute = ['EColi', 'CrVI']

# List of numeric columns to impute
other_num_cols = [col for col in wqi_data_convert_cell.columns if col not in num_cols_to_impute]

# Create a copy to avoid modifying the original
df = wqi_data_convert_cell.copy()

# Select columns for KNN to work on
imputer = KNNImputer(n_neighbors=5)

df[num_cols_to_impute] = imputer.fit_transform(
    df[num_cols_to_impute + other_num_cols]
)[:, :len(num_cols_to_impute)]

wqi_data_convert_cell['EColi'] = df['EColi']
wqi_data_convert_cell['CrVI'] = df['CrVI']

I choose KNN because it better preserves the multivariate relationships between water quality parameters, making the imputed values more realistic. It also maintains the variance and distribution of the data, avoiding the loss of extreme values that are important for WQI. KNN improves machine learning model performance by providing context-aware imputations and is well-suited to the environmental similarity inherent in monitoring data. Overall, it leads to more accurate and reliable WQI predictions.

In [ ]:
wqi_data_convert_cell.isnull().sum()

All missing values have already been handled.

## 1.3 Removing Inconsistent values and Outliers


In [ ]:
numeric_cols = wqi_data_convert_cell.select_dtypes(include='number').columns
n_cols = len(numeric_cols)

plt.figure(figsize=(10, 4 * n_cols))

for i, col in enumerate(numeric_cols):
    ax = plt.subplot(n_cols, 1, i + 1)
    sns.boxplot(
        y=wqi_data_convert_cell[col],
        ax=ax,
        color="#4c72b0",
        width=0.4
    )
    ax.set_title(col, fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("Value")

plt.tight_layout()
plt.show()

Although there are many outliers in the data, they are likely to be false positive noise or valid extreme observations, so I decided not to handle them.

- Temperature (°C)
The temperature values show a moderately concentrated distribution, with most observations falling within a relatively narrow range around the median. The median temperature is approximately in the mid-20°C range, indicating generally warm water conditions. The interquartile range suggests moderate variability, reflecting some natural fluctuations in temperature across samples. A small number of lower-end outliers can be observed, which may correspond to measurements taken under cooler environmental conditions or at different times. Overall, the distribution indicates that water temperature is fairly stable, with no extreme variability that would suggest abnormal thermal conditions.

- pH
The pH values exhibit a relatively narrow distribution, with most observations clustered around the neutral range. The median pH is approximately between 6.5 and 7.0, indicating near-neutral water conditions that are generally suitable for aquatic life. The interquartile range is small, suggesting limited variability in pH across the sampled locations or time periods. However, several outliers are observed at both the lower and upper ends of the distribution. Lower-end outliers may indicate localized acidic conditions, while higher-end outliers suggest slightly alkaline environments in certain samples. Overall, the pH distribution reflects relatively stable water quality, with occasional deviations that may be influenced by local environmental or anthropogenic factors.

- DO (Dissolved Oxygen) (mg/L)
The dissolved oxygen (DO) values show a moderately concentrated distribution, with the majority of observations lying between approximately 6 and 7 mg/L. The median DO level is around 6.5 mg/L, indicating generally adequate oxygen availability to support aquatic organisms. The interquartile range suggests moderate variability in DO concentrations across the samples. Several lower-end outliers are present, which may reflect localized oxygen depletion due to organic pollution, high biological oxygen demand, or stagnant water conditions. A few higher-end outliers are also observed, possibly associated with increased aeration or photosynthetic activity. Overall, the DO distribution suggests that water quality is generally suitable for aquatic life, although some sites or periods may experience reduced oxygen levels that warrant further attention.

- COD (Chemical Oxygen Demand) (mg/L)
The COD values show a relatively wide distribution with a noticeable number of outliers at higher concentrations. Most observations are concentrated at low to moderate levels, indicating acceptable baseline organic content in the majority of samples. However, several extreme values are observed, suggesting episodes of elevated organic pollution at specific locations or time periods. These high COD outliers may be attributed to industrial discharge, sewage contamination, or agricultural runoff. This suggests that while the general water quality is acceptable in many samples, certain locations or periods may experience significant organic contamination that requires further investigation and management.

- BOD₅ (Biochemical Oxygen Demand) (mg/L)
The BOD₅ distribution is moderately spread, with the majority of values clustered at lower concentrations, indicating relatively low levels of biodegradable organic matter in most samples. The median value falls within an acceptable range for aquatic ecosystems. However, the presence of several high outliers indicates increased biodegradable organic matter in some samples. These elevated BOD₅ values may reflect localized organic pollution or wastewater influence from domestic or industrial sources. Such contamination could potentially lead to higher oxygen demand and stress on aquatic ecosystems if not properly managed.

- NO₃⁻ (Nitrate) (mg/L)
The NO₃⁻ concentrations exhibit a generally narrow distribution at low levels, with most values clustered near the lower end of the scale. This pattern suggests that nitrate levels are typically low and stable across most samples, indicating limited nutrient loading under normal conditions. However, several clear outliers extending to higher values are observed. These occasional spikes may occur due to agricultural runoff, fertilizer use, or other nutrient inputs from anthropogenic activities. Such elevated nitrate concentrations could pose a risk of eutrophication if they persist or increase over time, warranting continued monitoring and potential management interventions.

- NO₂⁻ (Nitrite) (mg/L)
The NO₂⁻ values show a very narrow distribution, with most observations concentrated close to zero, indicating minimal nitrite presence in the majority of water samples. The median and interquartile range are both very low, reflecting stable nitrogen cycling processes in most locations. However, several clear outliers appear at higher concentrations, deviating significantly from the typical baseline. This indicates that nitrite levels are generally low and stable in most water samples, while occasional spikes may occur due to incomplete nitrification processes or localized pollution sources. Although infrequent, these elevated values may pose potential risks to aquatic life and should be monitored closely.

- NH₄⁺ (Ammonium) (mg/L)
The NH₄⁺ distribution is relatively narrow around low concentrations, with most values clustered near the baseline level. The median ammonium concentration is low, indicating minimal ammonia pollution in the majority of samples. However, a noticeable number of high outliers are present, extending well beyond the upper whisker of the boxplot. This suggests that ammonium concentrations are typically low across most samples, indicating generally good water quality. Nevertheless, the presence of extreme values may reflect episodic inputs from wastewater discharge, agricultural runoff, or organic matter decomposition, which could negatively impact water quality if persistent.

- PO₄³⁻ (Phosphate) (mg/L)
The PO₄³⁻ values are highly concentrated near zero, with a very narrow interquartile range, indicating minimal phosphate enrichment in most water samples. The median concentration is very low, suggesting limited anthropogenic phosphorus inputs under typical conditions. A few distinct outliers at higher concentrations are observed, indicating sporadic increases in phosphate levels at certain locations or times. These elevated values may be associated with fertilizer runoff or domestic wastewater inputs from urban or agricultural areas. Such phosphate spikes could contribute to eutrophication if they occur frequently, although overall phosphate concentrations remain low in most samples, suggesting limited nutrient enrichment under normal conditions.

- Coliform (MPN/100 mL)
The Coliform values exhibit a highly skewed distribution with a large number of extreme outliers extending far beyond the upper whisker. The median and interquartile range suggest that baseline coliform levels are relatively controlled in many samples. However, while most samples show relatively low concentrations, several observations reach very high levels, indicating significant variability in microbial contamination across the dataset. These elevated values suggest that certain locations or periods may be strongly affected by fecal pollution or untreated wastewater inputs. Such contamination poses potential risks to water quality and public health, necessitating targeted intervention and continuous microbiological monitoring.

- E. coli (MPN/100 mL)
The E. coli distribution is strongly right-skewed, with the majority of values clustered near zero and multiple high outliers. The median E. coli count is very low, suggesting that most water samples are relatively free from fecal contamination under baseline conditions. This pattern indicates that E. coli contamination is generally low in most samples, but sporadic contamination events occur. Such peaks may be associated with sewage discharge, surface runoff, or animal waste from agricultural activities. These findings highlight the need for regular microbiological monitoring to ensure water safety and protect public health.

- As (Arsenic) (mg/L)
The arsenic (As) concentrations show a narrow distribution at low levels, with the median value close to zero or very minimal. Most samples fall within a tight interquartile range, indicating that arsenic contamination is limited in the majority of locations. However, a small number of higher outliers are present, suggesting occasional exceedances of typical background levels. This suggests that arsenic levels are typically low and stable in most water samples. The presence of outliers indicates occasional exceedances that may be linked to geological sources or anthropogenic activities such as mining or industrial processes. These elevated arsenic concentrations warrant attention due to potential long-term health risks associated with chronic exposure.

- Hg (Mercury) (mg/L)
The distribution of Hg concentrations is highly skewed, with the majority of values clustered very close to zero, indicating generally low background levels in most water samples. The interquartile range is extremely narrow, suggesting minimal mercury presence under typical environmental conditions. However, several distinct outliers are observed at higher concentrations, deviating markedly from the baseline. These outliers suggest sporadic mercury contamination events, which may be associated with localized pollution sources or episodic discharges from industrial activities. Although typical Hg levels appear low and stable, the presence of extreme values is environmentally significant and should not be overlooked due to mercury's high toxicity even at low concentrations.

- Pb (Lead) (mg/L)
Pb exhibits a wider distribution compared to Hg, with a slightly higher median and a more dispersed interquartile range. The central tendency indicates that lead is present at detectable levels in many samples, though concentrations remain generally low. A substantial number of outliers are present, including several extreme values far above the upper whisker. This pattern indicates considerable variability in lead concentrations across sampling locations or time periods. The frequent occurrence of high outliers suggests that lead pollution is not isolated but may be influenced by multiple anthropogenic sources such as industrial effluents, traffic-related runoff, or legacy contamination. This variability highlights Pb as a critical parameter requiring close monitoring.

- Cd (Cadmium) (mg/L)
The Cd box plot shows a narrow central distribution with most values concentrated at low concentrations, similar to Hg. The median cadmium level is very low, indicating minimal cadmium presence in the majority of water samples. Nevertheless, several pronounced outliers are visible, including at least one extreme value significantly higher than the rest of the dataset. This indicates that while cadmium levels are generally low and stable in most samples, occasional spikes occur. Such spikes may reflect localized contamination or abnormal environmental conditions from industrial sources or mining activities. Given the high ecological and health risks associated with cadmium, these outliers are of particular concern despite the low median concentration.

- Cr(VI) (Hexavalent Chromium) (mg/L)
The distribution of Cr(VI) concentrations is highly right-skewed, with the vast majority of values tightly clustered near zero, indicating generally low and stable background levels in most water samples. The interquartile range is very narrow, suggesting limited variability under normal conditions. However, several distinct outliers are observed, including at least one extreme value that is markedly higher than the rest of the dataset. These outliers indicate sporadic but significant chromium contamination events, potentially linked to localized industrial activities or accidental discharges from electroplating, leather tanning, or chemical manufacturing. Despite the low median concentration, the presence of extreme Cr(VI) values is environmentally critical due to the high toxicity of hexavalent chromium, warranting careful monitoring and targeted management.

## 1.3 Caculate the WQI output

---

The WQI was calculated using the formula provided by the Vietnamese government.

In [ ]:
# 1. Configure the BP threshold values for each metal
# The order of values corresponds to i = 1, 2, 3, 4, 5
metal_configs = {
    'As_mg_L':   [0.01, 0.02, 0.05, 0.1,  0.1], 
    'Cd_mg_L':   [0.005, 0.005, 0.008, 0.01, 0.1],
    'Pb_mg_L':   [0.02, 0.02, 0.04, 0.05, 0.5],
    'CrVI': [0.01, 0.02, 0.04, 0.05, 0.1],
    'Cu':   [0.1, 0.2, 0.5, 1.0, 2.0],
    'Zn':   [0.5, 1.0, 1.5, 2.0, 3.0],
    'Hg_mg_L':   [0.001, 0.001, 0.0015, 0.002, 0.01]
}

# q values corresponding to levels i from 1 to 5
q_values = [100, 75, 50, 25, 10]

# 2. General WQI calculation function
def calculate_wqi_generic(cp, bp_list):
    if pd.isna(cp) or not isinstance(cp, (int, float)):
        return np.nan

    # If Cp is better than or equal to the highest standard level (level 1)
    if cp <= bp_list[0]:
        return q_values[0]

    # If Cp is worse than or equal to the lowest standard level (level 5)
    if cp >= bp_list[4]:
        return q_values[4]

    # Find the interval to apply Formula 1
    for i in range(len(bp_list) - 1):
        bp_i, bp_i1 = bp_list[i], bp_list[i+1]
        q_i, q_i1 = q_values[i], q_values[i+1]

        if bp_i <= cp <= bp_i1:
            # Case when bp_i == bp_i1 (like Pb, Hg at levels 1-2)
            if cp == bp_i and i != 0:
                return cp
            if cp == bp_i1 and i != 4:
                return cp

            wqi = ((q_i - q_i1) / (bp_i1 - bp_i)) * (bp_i1 - cp) + q_i1
            return round(wqi, 2)

# 3. Loop to automatically calculate for all columns in the configuration
for metal, bp_points in metal_configs.items():
    if metal in wqi_data_convert_cell.columns:
        new_col_name = f'WQI_{metal.split()[0]}'
        wqi_data_convert_cell[new_col_name] = wqi_data_convert_cell[metal].apply(
            lambda x: calculate_wqi_generic(x, bp_points)
        )

# Display the updated DataFrame with new WQI columns
print(wqi_data_convert_cell.head())

In [ ]:
def calculate_wqi_do(do_measured, temp):
    """
    Calculate the WQI for the DO parameter based on temperature and dissolved oxygen concentration.
    do_measured: Column 'DO mg/L'
    temp: Column 'Temperature (°C)'
    """
    if pd.isna(do_measured) or pd.isna(temp) or not isinstance(do_measured, (int, float)) or not isinstance(temp, (int, float)):
        return np.nan

    # Step 1: Calculate saturated DO (DO_saturated) based on temperature T
    # Formula: 14.652 - 0.41022*T + 0.0079910*T^2 - 0.0000777747*T^3
    do_sat = (14.652 - 0.41022 * temp +
              0.0079910 * (temp**2) -
              0.000077774 * (temp**3))

    # Calculate DO % saturation (Cp)
    cp = (do_measured / do_sat) * 100

    # Table 3: Standard values of BPi and qi for DO % saturation
    bp_do = [20, 50, 75, 88, 112, 125, 150, 200]
    q_do = [25, 50, 75, 100, 100, 75, 50, 25]

    # Check if Cp exactly matches any breakpoint values in the table
    for i, bp_val in enumerate(bp_do):
        if round(cp, 2) == bp_val:
            return q_do[i]

    if cp < 20 or cp > 200:
        return 10

    if 88 <= cp <= 112:
        return 100

    # Case 20 < Cp < 88: Use Formula 2
    if 20 < cp < 88:
        # Determine the interval i
        for i in range(0, 3): # Only consider breakpoints 20, 50, 75, 88
            bp_i, bp_i1 = bp_do[i], bp_do[i+1]
            q_i, q_i1 = q_do[i], q_do[i+1]

            if bp_i < cp < bp_i1:
                # Formula 2: WQI = [(qi+1 - qi) / (BPi+1 - BPi)] * (Cp - BPi) + qi
                wqi = ((q_i1 - q_i) / (bp_i1 - bp_i)) * (cp - bp_i) + q_i
                return round(wqi, 2)

    # Case 112 < Cp < 200: Use Formula 1
    if 112 < cp < 200:
        # Determine the interval i (breakpoints 112, 125, 150, 200)
        for i in range(4, 7):
            bp_i, bp_i1 = bp_do[i], bp_do[i+1]
            q_i, q_i1 = q_do[i], q_do[i+1]

            if bp_i < cp < bp_i1:
                # Formula 1: WQI = [(qi - qi+1) / (BPi+1 - BPi)] * (BPi+1 - Cp) + qi+1
                wqi = ((q_i - q_i1) / (bp_i1 - bp_i)) * (bp_i1 - cp) + q_i1
                return round(wqi, 2)


# Apply to DataFrame
wqi_data_convert_cell['WQI_DO'] = wqi_data_convert_cell.apply(
    lambda row: calculate_wqi_do(row['DO_mg_L'], row['Temperature_C']), axis=1
)

print(wqi_data_convert_cell[['DO_mg_L', 'Temperature_C', 'WQI_DO']].head())

In [ ]:
def calculate_wqi_ph(cp):
    """
    Calculate the WQI for the pH parameter based on Table 4 and interpolation formulas.
    cp: observed pH value.
    """

    if pd.isna(cp) or not isinstance(cp, (int, float)):
        return np.nan

    # Table 4: Regulations for BPi and qi values for pH
    # i=1: <5.5 (q=10), i=2: 5.5 (q=50), i=3: 6 (q=100)
    # i=4: 8.5 (q=100), i=5: 9 (q=50), i=6: >9 (q=10)

    # 1. Case when the value is outside the range or exactly at an extreme boundary
    if cp < 5.5 or cp > 9:
        return 10


    # 2. Case when the value exactly matches the breakpoint values in the table
    if cp == 5.5: return 50
    if cp == 6.0: return 100
    if cp == 8.5: return 100
    if cp == 9.0: return 50

    # 3. Case when 5.5 < pH < 6: Use Formula 2
    if 5.5 < cp < 6:
        # i=2 (BPi=5.5, qi=50); i+1=3 (BPi+1=6, qi+1=100)
        bp_i, bp_i1 = 5.5, 6.0
        q_i, q_i1 = 50, 100
        # WQI = [(qi+1 - qi) / (BPi+1 - BPi)] * (Cp - BPi) + qi
        wqi = ((q_i1 - q_i) / (bp_i1 - bp_i)) * (cp - bp_i) + q_i
        return round(wqi, 2)

    # 4. Case when 6 <= pH <= 8.5: WQI = 100
    if 6 <= cp <= 8.5:
        return 100

    # 5. Case when 8.5 < pH < 9: Use Formula 1
    if 8.5 < cp < 9:
        # i=4 (BPi=8.5, qi=100); i+1=5 (BPi+1=9, qi+1=50)
        bp_i, bp_i1 = 8.5, 9.0
        q_i, q_i1 = 100, 50
        # WQI = [(qi - qi+1) / (BPi+1 - BPi)] * (BPi+1 - Cp) + qi+1
        wqi = ((q_i - q_i1) / (bp_i1 - bp_i)) * (bp_i1 - cp) + q_i1
        return round(wqi, 2)

# Apply to the entire data in the 'pH' column
wqi_data_convert_cell['WQI_ph'] = wqi_data_convert_cell['pH'].apply(calculate_wqi_ph)

# Display the check results
print(wqi_data_convert_cell[['pH', 'WQI_ph']].head())

In [ ]:
# 1. Set up the BP benchmark values table for each parameter
# Corresponding q values order: 100, 75, 50, 25, 10
q_values = [100, 75, 50, 25, 10]

params_config = {
    'BOD5_mg_L':          [4, 6, 15, 25, 50],
    'COD_mg_L':           [10, 15, 30, 50, 150],
    'TOC':                [4, 6, 15, 25, 50],
    'NH4_mg_L':          [0.3, 0.3, 0.6, 0.9, 5], 
    'NO3_mg_L':           [2, 5, 10, 15, 15],  
    'PO4_mg_L':         [0.1, 0.2, 0.3, 0.5, 4],
    'Coliform_MPN_100ml': [2500, 5000, 7500, 10000, 10000],
    'EColi':             [20, 50, 100, 200, 200]
}

# 2. General function to calculate WQI SI (using Formula 1)
def calculate_wqi_formula1(cp, bp_list):
    if pd.isna(cp): return np.nan

    # Case when the value matches a boundary or exceeds an extreme limit
    if cp <= bp_list[0]: return q_values[0]
    if cp >= bp_list[4]: return q_values[4]

    # Check if the Cp value exactly matches the BPi benchmark point
    for i, bp_val in enumerate(bp_list):
        if cp == bp_val:
            return q_values[i]

    # Apply Formula 1 for the intervals
    for i in range(len(bp_list) - 1):
        bp_i, bp_i1 = bp_list[i], bp_list[i+1]
        q_i, q_i1 = q_values[i], q_values[i+1]

        if bp_i < cp < bp_i1:
            # WQI = [(qi - qi+1) / (BPi+1 - BPi)] * (BPi+1 - Cp) + qi+1
            wqi = ((q_i - q_i1) / (bp_i1 - bp_i)) * (bp_i1 - cp) + q_i1
            return round(wqi, 2)

# 3. Loop automatically to calculate for columns (Except NO2- handled separately)
for param, bp_points in params_config.items():
    if param in wqi_data_convert_cell.columns:
        # Create a new column name, e.g., WQI_BOD5, WQI_NH4...
        new_col = f"WQI_{param.replace('+', '').replace('-', '')}"
        wqi_data_convert_cell[new_col] = wqi_data_convert_cell[param].apply(
            lambda x: calculate_wqi_formula1(x, bp_points)
        )

# 4. Handle NO2- separately (Because the table only has 2 levels: <=0.05 and >0.05)
if 'NO2_mg_L' in wqi_data_convert_cell.columns:
    wqi_data_convert_cell['WQI_NO2_mg_L'] = wqi_data_convert_cell['NO2_mg_L'].apply(
        lambda x: 100 if x <= 0.05 else 10 if x > 0.05 else np.nan
    )

# Display the results of the newly created WQI columns
wqi_cols = [c for c in wqi_data_convert_cell.columns if c.startswith('WQI_')]
print(wqi_data_convert_cell[wqi_cols].head())

In [ ]:
wqi_data_convert_cell.isnull().sum()

In [ ]:
def calculate_final_vn_wqi(row):
    # 1. Grouping of parameters
    group_i_cols = ['WQI_ph']

    # Group II: Pesticides
    group_ii_cols = [col for col in row.index if col.startswith('WQI_') and
                     any(p in col for p in ['Aldrin', 'DDTs', 'BHC', 'Dieldrin'])]

    # Group III: Heavy metals
    group_iii_cols = ['WQI_As_mg_L', 'WQI_Cd_mg_L', 'WQI_Pb_mg_L', 'WQI_CrVI', 'WQI_Cu', 'WQI_Zn', 'WQI_Hg_mg_L']

    # Group IV: Organic & Nutrients (Mandatory - at least 3 parameters)
    group_iv_cols = ['WQI_DO', 'WQI_BOD5_mg_L', 'WQI_COD_mg_L', 'WQI_TOC', 'NH4_mg_L', 'NO3_mg_L', 'WQI_NO2_mg_L', 'WQI_PO4_mg_L']

    # Group V: Microbiological
    group_v_cols = ['WQI_Coliform_MPN_100ml', 'WQI_EColi']

    # 2. Check calculation conditions
    # - Group IV is mandatory and must have at least 3 parameters
    valid_iv = [row[c] for c in group_iv_cols if c in row.index and pd.notna(row[c])]
    if len(valid_iv) < 3:
        return np.nan # Not enough conditions to calculate VN_WQI
    # - At least 3 out of 5 parameter groups must be available 
    available_groups = 0
    groups_data = {}

    for g_name, cols in [('I', group_i_cols), ('II', group_ii_cols),
                         ('III', group_iii_cols), ('IV', group_iv_cols), ('V', group_v_cols)]:
        valid_vals = [row[c] for c in cols if c in row.index and pd.notna(row[c])]
        if valid_vals:
            available_groups += 1
            groups_data[g_name] = valid_vals

    if available_groups < 3:
        return np.nan

    # 3. Calculate the average value for each group
    # Group I: WQI_I
    wqi_1 = groups_data['I'][0] if 'I' in groups_data else 100

    # Group II & III: Geometric Mean
    wqi_2_val = gmean(groups_data['II']) if 'II' in groups_data else 100
    wqi_3_val = gmean(groups_data['III']) if 'III' in groups_data else 100

    # Group IV & V: Arithmetic Mean
    wqi_4_avg = np.mean(groups_data['IV'])
    wqi_5_avg = np.mean(groups_data['V']) if 'V' in groups_data else np.nan

    # 4. Apply Formula 3
    # Square root component of groups IV and V
    if not np.isnan(wqi_5_avg):
        bracket_term = np.sqrt(wqi_4_avg * wqi_5_avg)
    else:
        bracket_term = wqi_4_avg # If group V is missing, use only group IV
    final_wqi = (wqi_1 / 100) * (wqi_2_val / 100) * (wqi_3_val / 100) * bracket_term

    return round(final_wqi, 0)

# Apply calculation to the entire DataFrame
wqi_data_convert_cell['VN_WQI'] = wqi_data_convert_cell.apply(calculate_final_vn_wqi, axis=1)

# Display the final results
print(wqi_data_convert_cell[['VN_WQI']].head())

In [ ]:
wqi_data_convert_cell.info()

In [ ]:
final_data = wqi_data_convert_cell.copy()

Intermediate WQI sub-index columns, which were used solely for calculating the final VN_WQI score according to the Vietnamese government's formula, have been removed. 
The final dataset retains only the raw water quality parameters (input features) and the computed VN_WQI (target variable), allowing machine learning models to learn direct relationships between measurements and the water quality index. 
In future studies, WQI sub-indices could also be used as input features instead of the raw parameters.

In [ ]:
numeric_df = final_data.select_dtypes(include='number')

stats_table = pd.DataFrame({
    'Minimum': numeric_df.min(),
    'Maximum': numeric_df.max(),
    'Mean': numeric_df.mean(),
    'Median': numeric_df.median(),
    'Std. Deviation': numeric_df.std()
})

stats_table = stats_table.round(3)

stats_table.style.format("{:.3f}") \
    .set_caption("Descriptive Statistics of Water Quality Parameters") \
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '14px'), ('font-weight', 'bold')]},
        {'selector': 'th',
         'props': [('background-color', "#000000"),
                   ('font-weight', 'bold'),
                   ('text-align', 'center')]},
        {'selector': 'td',
         'props': [('text-align', 'center')]}
    ])

Final dataset description: 
- Water quality data from Quang Ninh Province, Vietnam, were collected from 2019 to 2021 at multiple monitoring stations covering rivers, lakes, and coastal areas, including key sites near Ha Long Bay, a UNESCO World Natural Heritage site. The dataset includes monthly and bimonthly measurements of physical, chemical, microbiological, and heavy metal parameters to provide a comprehensive assessment of water quality in the province.
- The monitored water quality variables comprised: Temperature (°C), pH, Dissolved Oxygen (DO, mg/L), Biochemical Oxygen Demand over 5 days (BOD₅, mg/L), Chemical Oxygen Demand (COD, mg/L), Nitrate (NO₃, mg/L), Nitrite (NO₂, mg/L), Ammonium (NH₄⁺, mg/L), Phosphate (PO₄³⁻, mg/L), Coliform (MPN/100 mL), E. coli, and trace heavy metals (As, Hg, Pb, Cd, CrVI, mg/L).
- The descriptive statistics of the observed water quality parameters and WQI are summarized in Table 1. Parameters such as Coliform, COD, and BOD₅ exhibited considerable variability, reflecting differences in point and non-point sources of pollution, as well as seasonal hydro-climatic influences. In contrast, parameters such as pH and trace metals showed relatively stable values over the monitoring period.
- The Water Quality Index (WQI) was calculated for each monitoring station using standard guidelines issued by the Ministry of Natural Resources and Environment (MONRE) of Vietnam. The WQI provides an aggregated assessment of water quality, allowing classification into five categories: excellent (91–100), good (76–90), fair (51–75), poor (26–50), and very poor (0–25).

# 2. Exploratory Data Analysis


---

In [ ]:
numeric_cols = final_data.select_dtypes(include=['int64', 'float64']).columns
n_rows = len(numeric_cols)

plt.figure(figsize=(8, 4 * n_rows))

for i, col in enumerate(numeric_cols):
    plt.subplot(n_rows, 1, i + 1)
    sns.histplot(
        data=final_data,
        x=col,
        bins=15,
        kde=True
    )
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Get numeric columns
numeric_cols = final_data.select_dtypes(include=['int64', 'float64']).columns
df_numeric = final_data[numeric_cols]

# Calculate correlation matrix
corr_matrix = df_numeric.corr()

plt.figure(figsize=(24, 16))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Heatmap: Relationship between numeric variables in WQI dataset")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=final_data, x='Temperature_C', y='DO_mg_L',s=20)
plt.title('Compare between Temperature and DO')
plt.xlabel('Temperature (°C)')
plt.ylabel('DO (mg/l)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=final_data, x='COD_mg_L', y='BOD5_mg_L',s=20)
plt.title('Compare between COD and BOD5')
plt.xlabel('COD (mg/l)')
plt.ylabel('BOD5 (mg/l)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=final_data, x='Coliform_MPN_100ml', y='EColi',s=20)
plt.title('Compare between Coliform and Ecoli')
plt.xlabel('Coliform (MPN/100mL)')
plt.ylabel('Ecoli (MPN/100mL)')
plt.legend()
plt.show()

# 3. Feature Engineering


---

In [ ]:
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

knn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

svr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

Standard scaling is applied for Linear Regression and K-Nearest Neighbors because these models are sensitive to the scale of features. On the other hand, Random Forest and Gradient Boosting do not require feature scaling, as they are tree-based models and are insensitive to the magnitude of the features.

# 4. Direct Measurement Inputs

## 4.1 Before applying feature selection

### 4.1.1. Classifier Training

#### Data Splitting

In [ ]:
# Separate X and y
X = final_data[['Temperature_C', 'pH', 'DO_mg_L', 'COD_mg_L', 'BOD5_mg_L', 
                'NO3_mg_L', 'NO2_mg_L', 'NH4_mg_L', 'PO4_mg_L', 
                'Coliform_MPN_100ml', 'EColi', 'As_mg_L', 'Hg_mg_L', 'Pb_mg_L', 
                'Cd_mg_L', 'CrVI']]  # Features
y = final_data['VN_WQI']  # WQI index

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Model Training

In [ ]:
def tune_random_forest(X, y, cv=5, n_jobs=-1, random_state=42):
    rf_param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 20, 40],
        'min_samples_leaf': [1, 2, 5],
        'max_features': ['sqrt', 0.7, 1.0]
    }

    rf = RandomForestRegressor(random_state=random_state)

    rf_grid_search = GridSearchCV(
        estimator=rf,
        param_grid=rf_param_grid,
        cv=cv,
        n_jobs=n_jobs
    )

    rf_grid_search.fit(X, y)

    best_rf_params = rf_grid_search.best_params_

    print("Best parameters for RandomForest:")
    print(best_rf_params)

    return best_rf_params

best_rf_params = tune_random_forest(X, y)

In [ ]:
def tune_gradient_boosting(X, y, cv=5, n_jobs=-1, random_state=42):
    gb_param_grid = {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7]
    }

    gb = GradientBoostingRegressor(random_state=random_state)

    gb_grid_search = GridSearchCV(
        estimator=gb,
        param_grid=gb_param_grid,
        cv=cv,
        n_jobs=n_jobs
    )

    gb_grid_search.fit(X, y)

    best_gb_params = gb_grid_search.best_params_

    print("Best parameters for GradientBoosting:")
    print(best_gb_params)

    return best_gb_params

best_gb_params = tune_gradient_boosting(X, y)

In [ ]:
def tune_knn(knn_model, X, y, cv=5, n_jobs=-1):
    knn_param_grid = {
        'knn__n_neighbors': [3, 5, 6, 7, 9],
        'knn__weights': ['uniform', 'distance'],
        'knn__metric': ['minkowski', 'euclidean', 'manhattan'],
        'knn__p': [1, 2]
    }

    knn_grid_search = GridSearchCV(
        estimator=knn_model,
        param_grid=knn_param_grid,
        cv=cv,
        n_jobs=n_jobs
    )

    knn_grid_search.fit(X, y)

    best_knn_params = knn_grid_search.best_params_

    print("Best parameters for KNN:")
    print(best_knn_params)

    return best_knn_params

best_knn_params = tune_knn(knn_model, X, y)

In [ ]:
def tune_svr(svr_model, X, y, cv=5, n_jobs=-1):
    param_grid = {
        'svr__C': [0.1, 1, 10, 100],
        'svr__epsilon': [0.01, 0.1, 0.2, 0.5],
        'svr__gamma': ['scale', 'auto', 0.01, 0.1, 1]
    }

    svr_grid_search = GridSearchCV(
        estimator=svr_model,
        param_grid=param_grid,
        cv=cv,
        n_jobs=n_jobs
    )

    svr_grid_search.fit(X, y)

    best_svr_params = svr_grid_search.best_params_

    print("Best parameters for SVR:")
    print(best_svr_params)

    return best_svr_params

best_svr_params = tune_svr(svr_model, X, y)

In [ ]:
def train_all_models(X_train, y_train, X_test, best_rf_params, best_gb_params,
                     lr_model, knn_model, best_knn_params, svr_model, best_svr_params):
    models = {}
    predictions = {}

    # Random Forest
    rf_model = RandomForestRegressor(**best_rf_params, random_state=42)
    rf_model.fit(X_train, y_train)
    models['RF'] = rf_model
    predictions['RF'] = rf_model.predict(X_test)

    # Gradient Boosting
    gb_model = GradientBoostingRegressor(**best_gb_params, random_state=42)
    gb_model.fit(X_train, y_train)
    models['GB'] = gb_model
    predictions['GB'] = gb_model.predict(X_test)

    # Linear Regression (Bagging)
    lr_bagging = BaggingRegressor(estimator=lr_model, n_estimators=200, random_state=42)
    lr_bagging.fit(X_train, y_train)
    models['LR'] = lr_bagging
    predictions['LR'] = lr_bagging.predict(X_test)

    # KNN
    knn_model = knn_model.set_params(**{
        'knn__n_neighbors': best_knn_params['knn__n_neighbors'],
        'knn__weights': best_knn_params['knn__weights'],
        'knn__metric': best_knn_params['knn__metric'],
        'knn__p': best_knn_params['knn__p']
    })
    knn_model.fit(X_train, y_train)
    models['KNN'] = knn_model
    predictions['KNN'] = knn_model.predict(X_test)

    # SVR (Bagging)
    svr_model = svr_model.set_params(**{
        'svr__C': best_svr_params['svr__C'],
        'svr__epsilon': best_svr_params['svr__epsilon'],
        'svr__gamma': best_svr_params['svr__gamma']
    })
    svr_bagging = BaggingRegressor(estimator=svr_model, n_estimators=200, random_state=42)
    svr_bagging.fit(X_train, y_train)
    models['SVR'] = svr_bagging
    predictions['SVR'] = svr_bagging.predict(X_test)

    return models, predictions

models, predictions = train_all_models(
    X_train, y_train, X_test,
    best_rf_params, best_gb_params,
    lr_model, knn_model, best_knn_params,
    svr_model, best_svr_params
)

rf_pred = predictions['RF']
gb_pred = predictions['GB']
lr_pred = predictions['LR']
knn_pred = predictions['KNN']
svr_pred = predictions['SVR']

In [ ]:
def evaluate_stacking(best_rf_params, knn_model, best_knn_params, lr_model, svr_model, best_svr_params, best_gb_params,
                      X_train, y_train, X_test, y_test, meta_model=None):
    rf_model = RandomForestRegressor(**best_rf_params, random_state=42)
    gb_model = GradientBoostingRegressor(**best_gb_params, random_state=42)
    lr_bagging = BaggingRegressor(estimator=lr_model, n_estimators=200, random_state=42)
    knn_model = knn_model.set_params(**{
        'knn__n_neighbors': best_knn_params['knn__n_neighbors'],
        'knn__weights': best_knn_params['knn__weights'],
        'knn__metric': best_knn_params['knn__metric'],
        'knn__p': best_knn_params['knn__p']
    })
    svr_model = svr_model.set_params(**{
        'svr__C': best_svr_params['svr__C'],
        'svr__epsilon': best_svr_params['svr__epsilon'],
        'svr__gamma': best_svr_params['svr__gamma']
    })
    svr_bagging = BaggingRegressor(estimator=svr_model, n_estimators=200, random_state=42)
    
    if meta_model is None:
        meta_model = LinearRegression()
    
    # Dictionary of models as in original code
    models = {
        'RF': rf_model,
        'KNN': knn_model,
        'LR': lr_bagging,
        'SVR': svr_bagging,
        'GB': gb_model
    }
    
    results = []
    model_names = list(models.keys())

    best_stacking_model = None
    best_rmse = float('inf')
    for r in range(2, len(model_names) + 1):
        for combo in itertools.combinations(model_names, r):
            estimators = [(name.lower(), models[name]) for name in combo]

            stacking = StackingRegressor(
                estimators=estimators,
                final_estimator=meta_model,
                n_jobs=-1
            )

            stacking.fit(X_train, y_train)
            preds = stacking.predict(X_test)

            mse = mean_squared_error(y_test, preds)
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(y_test, preds)
            r2 = r2_score(y_test, preds)

            results.append({
                'Base Models': " + ".join(combo),
                'Num Models': len(combo),
                'MSE': mse,
                'RMSE': rmse,
                'MAE': mae,
                'R2': r2
            })

            # Check for the best model
            if rmse < best_rmse:
                best_rmse = rmse
                best_stacking_model = stacking

    results_df = pd.DataFrame(results).sort_values(by='RMSE')

    return results_df, best_stacking_model

results_df, best_stacking_model = evaluate_stacking(
    best_rf_params=best_rf_params,
    knn_model=knn_model,
    best_knn_params=best_knn_params,
    lr_model=lr_model,
    svr_model=svr_model,
    best_svr_params=best_svr_params,
    best_gb_params=best_gb_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

#### Model Evaluation

In [ ]:
def evaluate_models(y_test, predictions_dict, decimal_places=4):
    """
    Evaluate multiple models and display results in a formatted table.
    
    Parameters:
    -----------
    y_test : array-like
        True target values
    predictions_dict : dict
        Dictionary with model names as keys and predictions as values
        Example: {'Random Forest': rf_pred, 'Linear Regression': lr_pred}
    decimal_places : int, default=4
        Number of decimal places to round the results
        
    Returns:
    --------
    pd.DataFrame
        DataFrame containing all evaluation metrics for each model
    """
    results = {
        'Model': [],
        'MSE': [],
        'RMSE': [],
        'MAE': [],
        'R²': []
    }
    
    for model_name, predictions in predictions_dict.items():
        results['Model'].append(model_name)
        results['MSE'].append(mean_squared_error(y_test, predictions))
        results['RMSE'].append(np.sqrt(mean_squared_error(y_test, predictions)))
        results['MAE'].append(mean_absolute_error(y_test, predictions))
        results['R²'].append(r2_score(y_test, predictions))
    
    comparison_df = pd.DataFrame(results)
    comparison_df = comparison_df.round(decimal_places)
    print(comparison_df.to_string(index=False))
    
    return comparison_df

# Use the function to evaluate all models
predictions_dict = {
    'Random Forest': rf_pred,
    'Gradient Boosting': gb_pred,
    'Linear Regression': lr_pred,
    'K-Nearest Neighbors': knn_pred,
    'Support Vector Regression': svr_pred
}

comparison_df = evaluate_models(y_test, predictions_dict, decimal_places=4)

In [ ]:
def plot_results_table(
    results_df,
    title="Stacking Models Performance Metrics",
    figsize_width=16,
    row_height=0.35,
    font_size=9,
    header_color="#40466e",
    alt_row_color="#f1f1f2",
    top_n=5
):
    top_results = results_df.nsmallest(top_n, 'RMSE')
    
    fig, ax = plt.subplots(
        figsize=(figsize_width, len(top_results) * row_height + 2)
    )
    ax.axis('off')

    tbl = ax.table(
        cellText=top_results.round(4).values,
        colLabels=top_results.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 0.95]
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(font_size)
    tbl.auto_set_column_width(col=list(range(len(top_results.columns))))

    for (i, j), cell in tbl.get_celld().items():
        cell.set_height(0.06)
        if i == 0:
            cell.set_facecolor(header_color)
            cell.set_text_props(weight='bold', color='white')
        else:
            cell.set_facecolor(alt_row_color if i % 2 == 0 else 'white')
            cell.set_edgecolor(header_color)

    fig.suptitle(f'{title} (Top {top_n})', fontsize=16, weight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

plot_results_table(results_df, top_n=5)

In [ ]:
def plot_parallel_coordinates_all_models(
    results_df, 
    y_test,
    rf_pred,
    gb_pred,
    lr_pred,
    knn_pred,
    svr_pred,
    title="All Models Performance Metrics",
    top_n=15
):
    df_plot = results_df.copy()
    
    individual_predictions = {
        'Random Forest': rf_pred,
        'Gradient Boosting': gb_pred,
        'Linear Regression': lr_pred,
        'K-Nearest Neighbors': knn_pred,
        'Support Vector Regression': svr_pred
    }
    
    individual_rows = []
    for model_name, predictions in individual_predictions.items():
        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        
        individual_rows.append({
            'Base Models': model_name,
            'Num Models': 1,
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2
        })
    
    df_individual = pd.DataFrame(individual_rows)
    df_plot = pd.concat([df_individual, df_plot], ignore_index=True)
    df_plot = df_plot.nsmallest(top_n, 'RMSE').reset_index(drop=True)
    
    import plotly.express as px
    colors = px.colors.qualitative.Plotly
    n_models = len(df_plot)
    color_mapping = [colors[i % len(colors)] for i in range(n_models)]
    
    fig = go.Figure(data=
        go.Parcoords(
            line=dict(
                color=list(range(n_models)),
                colorscale=[[i/(n_models-1), color_mapping[i]] for i in range(n_models)],
                showscale=False
            ),
            dimensions=[
                dict(
                    range=[df_plot['MSE'].min() * 0.95, df_plot['MSE'].max() * 1.05],
                    label='MSE',
                    values=df_plot['MSE'].tolist(),
                    tickformat='.2f'
                ),
                dict(
                    range=[df_plot['RMSE'].min() * 0.95, df_plot['RMSE'].max() * 1.05],
                    label='RMSE',
                    values=df_plot['RMSE'].tolist(),
                    tickformat='.2f'
                ),
                dict(
                    range=[df_plot['MAE'].min() * 0.95, df_plot['MAE'].max() * 1.05],
                    label='MAE',
                    values=df_plot['MAE'].tolist(),
                    tickformat='.2f'
                ),
                dict(
                    range=[df_plot['R2'].min() - 0.01, 1.0],
                    label='R²',
                    values=df_plot['R2'].tolist(),
                    tickformat='.4f'
                )
            ],
            labelangle=-45,
            labelside='bottom'
        )
    )
    
    fig.update_layout(
        title=dict(
            text=f"{title}<br><sub>Top {top_n} Models by RMSE</sub>",
            font=dict(size=16, family='Arial, sans-serif', color='#2c3e50'),
            x=0.5,
            xanchor='center'
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(size=11, family='Arial, sans-serif'),
        height=800,
        width=1400,
        margin=dict(l=250, r=350, t=120, b=100),
        hovermode='closest',
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=1.02,
            bgcolor="rgba(255, 255, 255, 0.95)",
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=9)
        ),
        xaxis=dict(showgrid=False),
        yaxis=dict(showgrid=False)
    )
    
    for i, row in df_plot.iterrows():
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=10, color=color_mapping[i]),
            legendgroup=row['Base Models'],
            showlegend=True,
            name=f"{i+1}. {row['Base Models'][:30]}",
            hovertemplate=f"<b>{row['Base Models']}</b><br>" +
                         f"Num Models: {int(row['Num Models'])}<br>" +
                         f"MSE: {row['MSE']:.4f}<br>" +
                         f"RMSE: {row['RMSE']:.4f}<br>" +
                         f"MAE: {row['MAE']:.4f}<br>" +
                         f"R²: {row['R2']:.4f}<br>" +
                         "<extra></extra>"
        ))
    
    return fig

fig_all_models_41 = plot_parallel_coordinates_all_models(
    results_df=results_df,
    y_test=y_test,
    rf_pred=rf_pred,
    gb_pred=gb_pred,
    lr_pred=lr_pred,
    knn_pred=knn_pred,
    svr_pred=svr_pred,
    title="4.1: All Models Performance Metrics - Individual vs Stacking Ensemble (Before Feature Selection)",
    top_n=15
)

fig_all_models_41.show()

### 4.1.2. Feature Importance

In [ ]:
def plot_permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, figsize=(10,6), color="skyblue"):
    """
    Calculate and plot permutation importance for a given model and test data.
    
    Parameters:
    - model: Trained model to evaluate
    - X_test: Test features (DataFrame)
    - y_test: Test labels
    - n_repeats: Number of permutation rounds (default=30)
    - random_state: Random seed for reproducibility (default=42)
    - figsize: Tuple for figure size (default=(10,6))
    - color: Bar color for the plot (default="skyblue")
    """
    # Calculate permutation importance
    perm_importance = permutation_importance(model, X_test, y_test, n_repeats=n_repeats, random_state=random_state)
    
    # Sort features by importance
    sorted_idx = perm_importance.importances_mean.argsort()
    
    # Plot
    plt.figure(figsize=figsize)
    plt.barh(
        X_test.columns[sorted_idx],
        perm_importance.importances_mean[sorted_idx],
        color=color,
        edgecolor="black"
    )
    plt.xlabel("Permutation Importance")
    plt.ylabel("Features")
    plt.title("Permutation Feature Importance")
    plt.tight_layout()
    plt.show()
    
    # Return permutation importance object if needed
    return perm_importance

perm_importance_result = plot_permutation_importance(best_stacking_model, X_test, y_test)

## 4.2. After applying feature selection

### 4.2.1. Classifier Training

#### Data Splitting

In [ ]:
# Separate X and y
X_retrain = final_data[['Temperature_C', 'pH', 'DO_mg_L', 'COD_mg_L', 
                'NO2_mg_L', 'PO4_mg_L', 
                'EColi',  'Hg_mg_L', 'Pb_mg_L', 'CrVI']]  # Features
y_retrain = final_data['VN_WQI']  # WQI index

# Split data
X_train_retrain, X_test_retrain, y_train_retrain, y_test_retrain = train_test_split(X_retrain, y_retrain, test_size=0.2, random_state=42)

#### Model Training

In [ ]:
best_rf_params_retrain = tune_random_forest(X_retrain, y_retrain)
print("\n ")
best_gb_params_retrain = tune_gradient_boosting(X_retrain, y_retrain)
print("\n ")
best_knn_params_retrain = tune_knn(knn_model, X_retrain, y_retrain)
print("\n ")
best_svr_params_retrain = tune_svr(svr_model, X_retrain, y_retrain)

In [ ]:
models_retrain, predictions_retrain = train_all_models(
    X_train_retrain, y_train_retrain, X_test_retrain,
    best_rf_params_retrain, best_gb_params_retrain,
    lr_model, knn_model, best_knn_params_retrain,
    svr_model, best_svr_params_retrain
)

rf_pred_retrain = predictions_retrain['RF']
gb_pred_retrain = predictions_retrain['GB']
lr_pred_retrain = predictions_retrain['LR']
knn_pred_retrain = predictions_retrain['KNN']
svr_pred_retrain = predictions_retrain['SVR']

In [ ]:
results_df_retrain, best_stacking_model_retrain = evaluate_stacking(
    best_rf_params=best_rf_params_retrain,
    knn_model=knn_model,
    best_knn_params=best_knn_params_retrain,
    lr_model=lr_model,
    svr_model=svr_model,
    best_svr_params=best_svr_params_retrain,
    best_gb_params=best_gb_params_retrain,
    X_train=X_train_retrain,
    y_train=y_train_retrain,
    X_test=X_test_retrain,
    y_test=y_test_retrain
)
sm_pred_retrain = best_stacking_model_retrain.predict(X_test_retrain)

### Model Evaluation

In [ ]:
# Evaluate all models using the function
predictions_dict_retrain = {
    'Random Forest': rf_pred_retrain,
    'Gradient Boosting': gb_pred_retrain,
    'Linear Regression': lr_pred_retrain,
    'K-Nearest Neighbors': knn_pred_retrain,
    'Support Vector Regression': svr_pred_retrain
}

comparison_df_retrain = evaluate_models(y_test_retrain, predictions_dict_retrain, decimal_places=4)

In [ ]:
plot_results_table(results_df_retrain)

In [ ]:
fig_all_models_42 = plot_parallel_coordinates_all_models(
    results_df=results_df_retrain,
    y_test=y_test_retrain,
    rf_pred=rf_pred_retrain,
    gb_pred=gb_pred_retrain,
    lr_pred=lr_pred_retrain,
    knn_pred=knn_pred_retrain,
    svr_pred=svr_pred_retrain,
    title="4.2: All Models Performance Metrics - Individual vs Stacking Ensemble (After Feature Selection)"
)

fig_all_models_42.show()

In [ ]:
def plot_wqi_comparison(y_test, y_pred):
    """
    Plot a comparison between actual WQI values and predicted values.

    Parameters:
    - y_test: Actual WQI values (Pandas Series or list)
    - y_pred: Predicted WQI values (list, numpy array, or Pandas Series)

    The function does not return anything, it only displays the plot.
    """
    plt.figure(figsize=(20, 10))
    plt.plot(y_test.values, label='Actual Values', marker='o')
    plt.plot(y_pred, label='Stacking Model Predictions', marker='h')
    plt.title('Comparison of Actual Values vs Predicted Values')
    plt.xlabel('Index')
    plt.ylabel('Water Quality Index (WQI)')
    plt.legend()
    plt.show()

plot_wqi_comparison(y_test_retrain, sm_pred_retrain)

### 4.2.2. Feature Importance

In [ ]:
plot_permutation_importance(best_stacking_model_retrain, X_test_retrain, y_test_retrain);

# 5. Feature Engineered Inputs (WQI_SI)


## 5.1 Before applying feature selection


### 5.1.1. Classifier Training


#### Data Splitting

In [ ]:
# Separate X and y
X_WQIInput = final_data[['WQI_ph', 'WQI_DO', 'WQI_COD_mg_L', 'WQI_BOD5_mg_L', 
                'WQI_NO3_mg_L', 'WQI_NO2_mg_L', 'WQI_NH4_mg_L', 'WQI_PO4_mg_L', 
                'WQI_Coliform_MPN_100ml', 'WQI_EColi', 'WQI_As_mg_L', 'WQI_Hg_mg_L', 'WQI_Pb_mg_L', 
                'WQI_Cd_mg_L', 'WQI_CrVI']]  # Features
y_WQIInput = final_data['VN_WQI']  # WQI index

# Split data
X_train_WQIInput, X_test_WQIInput, y_train_WQIInput, y_test_WQIInput = train_test_split(X_WQIInput, y_WQIInput, test_size=0.2, random_state=42)

#### Model Training

In [ ]:
best_rf_params_WQIInput = tune_random_forest(X_WQIInput, y_WQIInput)
print("\n ")
best_gb_params_WQIInput = tune_gradient_boosting(X_WQIInput, y_WQIInput)
print("\n ")
best_knn_params_WQIInput = tune_knn(knn_model, X_WQIInput, y_WQIInput)
print("\n ")
best_svr_params_WQIInput = tune_svr(svr_model, X_WQIInput, y_WQIInput)

In [ ]:
models_WQIInput, predictions_WQIInput = train_all_models(
    X_train_WQIInput, y_train_WQIInput, X_test_WQIInput,
    best_rf_params_WQIInput, best_gb_params_WQIInput,
    lr_model, knn_model, best_knn_params_WQIInput,
    svr_model, best_svr_params_WQIInput
)

rf_pred_WQIInput = predictions_WQIInput['RF']
gb_pred_WQIInput = predictions_WQIInput['GB']
lr_pred_WQIInput = predictions_WQIInput['LR']
knn_pred_WQIInput = predictions_WQIInput['KNN']
svr_pred_WQIInput = predictions_WQIInput['SVR']

- Tree-based ensemble models (Random Forest and Gradient Boosting) clearly outperform other models in terms of all evaluated metrics. Gradient Boosting achieves the lowest MSE and MAE, as well as the highest R² value, indicating superior predictive accuracy and robustness.

- Support Vector Regression (SVR) shows competitive performance, particularly with respect to MAE and R², suggesting that it is able to capture the underlying relationships in the data more effectively than simpler models.

- K-Nearest Neighbors (KNN) provides moderate predictive performance. While its RMSE and R² are acceptable, it is less accurate than ensemble methods, which may be due to its sensitivity to feature scaling and local data structure.

- Linear Regression performs the poorest among all evaluated models, reflected by substantially higher MSE and RMSE values and a lower R². This suggests that the relationships in the dataset are likely non-linear, which linear regression fails to fully capture.

Overall Observations:

Ensemble tree-based models are highly effective for this dataset, likely due to their ability to handle complex, non-linear interactions among features.

Simpler models such as linear regression and KNN may be useful for preliminary analysis or interpretation but are insufficient for high-accuracy predictions.

Model selection should balance predictive performance with interpretability and computational efficiency depending on the application.

In [ ]:
results_df_WQIInput, best_stacking_model_WQIInput = evaluate_stacking(
    best_rf_params=best_rf_params_WQIInput,
    knn_model=knn_model,
    best_knn_params=best_knn_params_WQIInput,
    lr_model=lr_model,
    svr_model=svr_model,
    best_svr_params=best_svr_params_WQIInput,
    best_gb_params=best_gb_params_WQIInput,
    X_train=X_train_WQIInput,
    y_train=y_train_WQIInput,
    X_test=X_test_WQIInput,
    y_test=y_test_WQIInput
)

#### Model Evaluation

In [ ]:
# Evaluate all models using the function
predictions_dict_WQIInput = {
    'Random Forest': rf_pred_WQIInput,
    'Gradient Boosting': gb_pred_WQIInput,
    'Linear Regression': lr_pred_WQIInput,
    'K-Nearest Neighbors': knn_pred_WQIInput,
    'Support Vector Regression': svr_pred_WQIInput
}

comparison_df_WQIInput = evaluate_models(y_test_WQIInput, predictions_dict_WQIInput, decimal_places=4)

In [ ]:
plot_results_table(results_df_WQIInput)

In [ ]:
fig_all_models_51 = plot_parallel_coordinates_all_models(
    results_df=results_df_WQIInput,
    y_test=y_test_WQIInput,
    rf_pred=rf_pred_WQIInput,
    gb_pred=gb_pred_WQIInput,
    lr_pred=lr_pred_WQIInput,
    knn_pred=knn_pred_WQIInput,
    svr_pred=svr_pred_WQIInput,
    title="5.1: All Models Performance Metrics - Individual vs Stacking Ensemble (WQI Input - Before Feature Selection)"
)

fig_all_models_51.show()

### 5.1.2. Feature Importance

In [ ]:
plot_permutation_importance(best_stacking_model_WQIInput, X_test_WQIInput, y_test_WQIInput);

## 5.2. After applying feature selection

### 5.2.1. Classifier Training

#### Data Splitting

In [ ]:
# Separate X and y
X_WQIInput_retrain = final_data[['WQI_ph', 'WQI_DO', 'WQI_COD_mg_L', 
                'WQI_NO2_mg_L', 'WQI_PO4_mg_L', 
                'WQI_EColi', 'WQI_Hg_mg_L', 'WQI_Pb_mg_L', 
                'WQI_CrVI']]
y_WQIInput_retrain = final_data['VN_WQI']  # WQI index
# Split data
X_train_WQIInput_retrain, X_test_WQIInput_retrain, y_train_WQIInput_retrain, y_test_WQIInput_retrain = train_test_split(X_WQIInput_retrain, y_WQIInput_retrain, test_size=0.2, random_state=42)

#### Model Training

In [ ]:
best_rf_params_WQIInput_retrain = tune_random_forest(X_WQIInput_retrain, y_WQIInput_retrain)
print("\n ")
best_gb_params_WQIInput_retrain = tune_gradient_boosting(X_WQIInput_retrain, y_WQIInput_retrain)
print("\n ")
best_knn_params_WQIInput_retrain = tune_knn(knn_model, X_WQIInput_retrain, y_WQIInput_retrain)
print("\n ")
best_svr_params_WQIInput_retrain = tune_svr(svr_model, X_WQIInput_retrain, y_WQIInput_retrain)

In [ ]:
models_WQIInput_retrain, predictions_WQIInput_retrain = train_all_models(
    X_train_WQIInput_retrain, y_train_WQIInput_retrain, X_test_WQIInput_retrain,
    best_rf_params_WQIInput_retrain, best_gb_params_WQIInput_retrain,
    lr_model, knn_model, best_knn_params_WQIInput_retrain,
    svr_model, best_svr_params_WQIInput_retrain
)

rf_pred_WQIInput_retrain = predictions_WQIInput_retrain['RF']
gb_pred_WQIInput_retrain = predictions_WQIInput_retrain['GB']
lr_pred_WQIInput_retrain = predictions_WQIInput_retrain['LR']
knn_pred_WQIInput_retrain = predictions_WQIInput_retrain['KNN']
svr_pred_WQIInput_retrain = predictions_WQIInput_retrain['SVR']

In [ ]:
results_df_WQIInput_retrain, best_stacking_model_WQIInput_retrain = evaluate_stacking(
    best_rf_params=best_rf_params_WQIInput_retrain,
    knn_model=knn_model,
    best_knn_params=best_knn_params_WQIInput_retrain,
    lr_model=lr_model,
    svr_model=svr_model,
    best_svr_params=best_svr_params_WQIInput_retrain,
    best_gb_params=best_gb_params_WQIInput_retrain,
    X_train=X_train_WQIInput_retrain,
    y_train=y_train_WQIInput_retrain,
    X_test=X_test_WQIInput_retrain,
    y_test=y_test_WQIInput_retrain
)
sm_pred_WQIInput_retrain = best_stacking_model_WQIInput_retrain.predict(X_test_WQIInput_retrain)

In order to optimize model performance and reduce redundancy, several features were excluded based on their relative importance, information overlap, and practical measurement considerations:

- Arsenic (As) and Cadmium (Cd): These variables were removed due to their limited influence on the predictive model.

- Ammonium (NH₄) and Nitrate (NO₃): Both features were excluded as their information is largely captured by Nitrite (NO₂), which exhibits a stronger effect on model performance.

- Biochemical Oxygen Demand (BOD): Although BOD demonstrates a higher predictive contribution than Chemical Oxygen Demand (COD), it was omitted because COD effectively encompasses the relevant information while being significantly faster to measure (BOD requires 5 days).

- Coliform: This feature was removed as its predictive content is already represented by E. coli, which offers better model performance.

This feature reduction strategy ensures the model is both efficient and interpretable, minimizing redundancy while maintaining predictive accuracy.

After retraining with a reduced feature set, the Stacking model exhibits slightly higher error metrics compared to the full-feature model, reflecting a minor reduction in predictive accuracy.

Despite this slight decrease in performance, the model still achieves high R² (0.957) and low error values, indicating that it remains reliable and acceptable for practical prediction tasks.

The feature reduction strategy provides a cost-efficient approach by eliminating variables that are either redundant or time-consuming to measure, without substantially compromising the model’s predictive capability.

Overall, the results demonstrate that careful feature selection can balance accuracy, efficiency, and practicality, making the reduced-feature Stacking model suitable for real-world applications.

#### Model Evaluation

In [ ]:
# Evaluate all models using the function
predictions_dict_WQIInput_retrain = {
    'Random Forest': rf_pred_WQIInput_retrain,
    'Gradient Boosting': gb_pred_WQIInput_retrain,
    'Linear Regression': lr_pred_WQIInput_retrain,
    'K-Nearest Neighbors': knn_pred_WQIInput_retrain,
    'Support Vector Regression': svr_pred_WQIInput_retrain
}

comparison_df_WQIInput_retrain = evaluate_models(y_test_WQIInput_retrain, predictions_dict_WQIInput_retrain, decimal_places=4)

In [ ]:
plot_results_table(results_df_WQIInput_retrain)

In [ ]:
fig_all_models_52 = plot_parallel_coordinates_all_models(
    results_df=results_df_WQIInput_retrain,
    y_test=y_test_WQIInput_retrain,
    rf_pred=rf_pred_WQIInput_retrain,
    gb_pred=gb_pred_WQIInput_retrain,
    lr_pred=lr_pred_WQIInput_retrain,
    knn_pred=knn_pred_WQIInput_retrain,
    svr_pred=svr_pred_WQIInput_retrain,
    title="5.2: All Models Performance Metrics - Individual vs Stacking Ensemble (WQI Input - After Feature Selection)"
)

fig_all_models_52.show()

In [ ]:
# Side-by-side scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Plot 1: Actual vs Predicted WQI
axes[0].scatter(y_test_WQIInput_retrain.values, sm_pred_WQIInput_retrain, alpha=0.6)
min_v1 = min(y_test_WQIInput_retrain.min(), sm_pred_WQIInput_retrain.min())
max_v1 = max(y_test_WQIInput_retrain.max(), sm_pred_WQIInput_retrain.max())
axes[0].plot([min_v1, max_v1], [min_v1, max_v1], 'r--', label='y = x')
axes[0].set_xlabel('Actual WQI (y_test_WQIInput_retrain)')
axes[0].set_ylabel('Predicted WQI (sm_pred_WQIInput_retrain)')
axes[0].set_title('Scatter Plot: Actual vs Predicted WQI')
axes[0].legend()
axes[0].grid(True)

# Plot 2: Manual vs Reduced-feature WQI
manual_index = final_data['VN_WQI']
ml_index = wqi_data_droped_v3['VN_WQI']
axes[1].scatter(manual_index, ml_index, alpha=0.6)
min_v2 = min(manual_index.min(), ml_index.min())
max_v2 = max(manual_index.max(), ml_index.max())
axes[1].plot([min_v2, max_v2], [min_v2, max_v2], 'r--', label='y = x')
axes[1].set_xlabel('Index of WQI before remove unimportant features(calculated manually)')
axes[1].set_ylabel('Index of WQI after remove unimportant features(calculated manually)')
axes[1].set_title('Scatter Plot: Index of WQI before vs after feature removal')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 5.2.2. Feature Importance

In [ ]:
plot_permutation_importance(best_stacking_model_WQIInput_retrain, X_test_WQIInput_retrain, y_test_WQIInput_retrain);

# 6. Traditional WQI Formula Evaluation

In [ ]:
wqi_data_droped_v3 = final_data.copy()
wqi_data_droped_v3.drop({'EColi', 'COD_mg_L', 'NH4_mg_L', 'NO3_mg_L', 'Cd_mg_L', 'As_mg_L', 'VN_WQI'}, axis=1, inplace=True)
for metal, bp_points in metal_configs.items():
    if metal in wqi_data_droped_v3.columns:
        new_col_name = f'WQI_{metal.split()[0]}'
        wqi_data_droped_v3[new_col_name] = wqi_data_droped_v3[metal].apply(
            lambda x: calculate_wqi_generic(x, bp_points)
        )
wqi_data_droped_v3['WQI_DO'] = wqi_data_droped_v3.apply(
    lambda row: calculate_wqi_do(row['DO_mg_L'], row['Temperature_C']), axis=1
)
wqi_data_droped_v3['WQI_ph'] = wqi_data_droped_v3['pH'].apply(calculate_wqi_ph)
for param, bp_points in params_config.items():
    if param in wqi_data_droped_v3.columns:
        # Create a new column name, e.g., WQI_BOD5, WQI_NH4...
        new_col = f"WQI_{param.replace('+', '').replace('-', '')}"
        wqi_data_droped_v3[new_col] = wqi_data_droped_v3[param].apply(
            lambda x: calculate_wqi_formula1(x, bp_points)
        )

# 4. Handle NO2- separately (Because the table only has 2 levels: <=0.05 and >0.05)
if 'NO2_mg_L' in wqi_data_droped_v3.columns:
    wqi_data_droped_v3['WQI_NO2_mg_L'] = wqi_data_droped_v3['NO2_mg_L'].apply(
        lambda x: 100 if x <= 0.05 else 10 if x > 0.05 else np.nan
    )
wqi_data_droped_v3['VN_WQI'] = wqi_data_droped_v3.apply(calculate_final_vn_wqi, axis=1)

In [ ]:
y1 = wqi_data_droped_v3['VN_WQI']
y2 = final_data['VN_WQI']
print("Retrained Data - MSE:", mean_squared_error(y2, y1))
print("Retrained Data - RMSE:", np.sqrt(mean_squared_error(y2, y1)))
print("Retrained Data - MAE:", mean_absolute_error(y2, y1))
print("Retrained Data - R^2:", r2_score(y2, y1))

The recalculated WQI demonstrates that, while it is technically possible to compute the index with fewer parameters, the resulting values deviate substantially from the original full-parameter WQI.

The negative R² indicates a very poor fit compared to the original WQI, highlighting that the reduced-parameter index fails to accurately reflect the water quality profile.

This significant discrepancy underscores the sensitivity of the traditional WQI method to the inclusion of key parameters. Omitting certain variables severely compromises its reliability and interpretability.

In practical applications, although the recalculated WQI remains formally acceptable, it may mislead assessments or decision-making in water quality monitoring.

These results emphasize the advantage of machine learning–based approaches, such as the Stacking Ensemble model, which maintain high predictive accuracy and robustness even when some features are omitted, providing a more reliable alternative for water quality evaluation.

In [ ]:
# Indices already visualized in the side-by-side scatter plots above
manual_index = final_data['VN_WQI']
ml_index = wqi_data_droped_v3['VN_WQI']

The scatter plot above compares the Water Quality Index (WQI) calculated using all original parameters against the WQI recalculated after removing unimportant features. The red dashed line represents the 1:1 line (y=x), where points would lie if the recalculated WQI perfectly matched the original full-parameter WQI.

- The scatter plot shows that most points lie above the 1:1 line, indicating that the recalculated WQI generally produces higher index values than the original WQI.

- The deviation from the 1:1 line is more pronounced at mid-range WQI values, suggesting that the reduction of features disproportionately inflates WQI scores for moderately polluted samples.

- Although the recalculated WQI remains formally valid and recognizable, this consistent upward shift highlights a systematic bias introduced by removing certain parameters.

- These results emphasize the sensitivity of the traditional WQI method to the inclusion of key water quality parameters and demonstrate the limitations of using a reduced subset for accurate water quality assessment.

- In practical terms, while WQI can still be computed with fewer parameters, the overestimation of water quality scores could mislead environmental assessments or decision-making if interpreted directly.